In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os


In [ ]:
dataset_path = "dataset/CDnet2014"
print("Dataset path set to:", dataset_path)

In [ ]:
def show_image(img, title="Image"):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(6,6))
    plt.imshow(img_rgb)
    plt.title(title)
    plt.axis("off")

# Automated Human Detection and Silhouette Segmentation in Surveillance Environments

Dataset: CDnet 2014

Pipeline:
1. Load surveillance frames
2. Gaussian smoothing (noise reduction)
3. Background subtraction (GMM)
4. Morphological operations
5. Human silhouette segmentation
6. Performance evaluation

In [ ]:
category = "baseline"
video = "highway"

input_path = f"{dataset_path}/{category}/{video}/input"

print("Input path:", input_path)

In [ ]:
import os

frames = sorted(os.listdir(input_path))

print("Total frames:", len(frames))
print("First 5 frames:", frames[:5])

In [ ]:
frame_path = os.path.join(input_path, frames[0])

frame = cv2.imread(frame_path)

print("Frame shape:", frame.shape)

In [ ]:
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6,6))
plt.imshow(frame_rgb)
plt.title("Original Frame")
plt.axis("off")

In [ ]:
display_frame = frame.copy()

In [ ]:
cv2.rectangle(display_frame, (200,150), (350,400), (0,255,0), 2)

In [ ]:
display_rgb = cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6,6))
plt.imshow(display_rgb)
plt.title("Human Detection (Example)")
plt.axis("off")

In [ ]:
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

print(gray.shape)

In [ ]:
blur = cv2.GaussianBlur(gray, (5,5), 0)

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(gray, cmap='gray')
plt.title("Original Grayscale")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(blur, cmap='gray')
plt.title("Gaussian Smoothed")
plt.axis("off")

In [ ]:
bg_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True
)

In [ ]:
fg_mask = bg_subtractor.apply(blur)

In [ ]:
plt.figure(figsize=(6,6))
plt.imshow(fg_mask, cmap='gray')
plt.title("Foreground Mask (GMM)")
plt.axis("off")

In [ ]:
kernel = np.ones((5,5), np.uint8)

In [ ]:
opening = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)

closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel)

In [ ]:
plt.figure(figsize=(6,6))
plt.imshow(closing, cmap='gray')
plt.title("Cleaned Foreground Mask")
plt.axis("off")

In [ ]:
contours, _ = cv2.findContours(
    closing,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

In [ ]:
result = frame.copy()

for cnt in contours:
    
    area = cv2.contourArea(cnt)
    
    if area > 1500:   # ignore small noise
        
        x, y, w, h = cv2.boundingRect(cnt)
        
        cv2.rectangle(result, (x,y), (x+w,y+h), (0,255,0), 2)

In [ ]:
result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6,6))
plt.imshow(result_rgb)
plt.title("Human Detection")
plt.axis("off")

In [ ]:
for i in range(0, 50):

    frame_path = os.path.join(input_path, frames[i])
    frame = cv2.imread(frame_path)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)

    fg_mask = bg_subtractor.apply(blur)
    opening = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)
    closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(closing, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:

        area = cv2.contourArea(cnt)

        if area > 1500:

            x,y,w,h = cv2.boundingRect(cnt)

            cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.imshow(frame_rgb)
    plt.axis("off")
    plt.show()

In [ ]:
gt_path = f"{dataset_path}/{category}/{video}/groundtruth"

print("Groundtruth path:", gt_path)

In [ ]:
gt_frame_path = os.path.join(gt_path, "gt000001.png")

gt_mask = cv2.imread(gt_frame_path, cv2.IMREAD_GRAYSCALE)

plt.figure(figsize=(6,6))
plt.imshow(gt_mask, cmap='gray')
plt.title("Ground Truth Mask")
plt.axis("off")

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.imshow(closing, cmap='gray')
plt.title("Predicted Mask")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(gt_mask, cmap='gray')
plt.title("Ground Truth")
plt.axis("off")

In [ ]:
pred = closing > 0
gt = gt_mask > 0

In [ ]:
TP = np.sum((pred == 1) & (gt == 1))
FP = np.sum((pred == 1) & (gt == 0))
FN = np.sum((pred == 0) & (gt == 1))
TN = np.sum((pred == 0) & (gt == 0))

print("TP:", TP)
print("FP:", FP)
print("FN:", FN)
print("TN:", TN)

In [ ]:
precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)

print("Precision:", precision)
print("Recall:", recall)

In [ ]:
f1 = 2 * (precision * recall) / (precision + recall + 1e-6)

print("F1 Score:", f1)

In [ ]:
precision_list = []
recall_list = []
f1_list = []

In [ ]:
precision_list.append(precision)
recall_list.append(recall)
f1_list.append(f1)

In [ ]:
plt.figure()

plt.plot(precision_list)
plt.title("Precision over Frames")
plt.xlabel("Frame")
plt.ylabel("Precision")
plt.show()

In [ ]:
plt.figure()

plt.plot(f1_list)
plt.title("F1 Score over Frames")
plt.xlabel("Frame")
plt.ylabel("F1 Score")
plt.show()